## Tokenizer avec Tiktoken

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o")
print(encoding.name)

In [ ]:
system_message = """
Perform Sentiment analysis of the review presented in the user message.
The result should be positive or negative. Do not justify your response
"""

tokens = encoding.encode(system_message)
print(len(tokens))
print(tokens)
for token in tokens:
    print(encoding.decode_single_token_bytes(token=token), end="")

In [ ]:
def num_tokens_from_string(string: str, encoding_name: str = "o200k_base") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string("tiktoken is great!")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from dotenv.ipython import load_dotenv
from IPython.display import display, Markdown
import os

load_dotenv(override=True)

In [ ]:
model = ChatOpenAI(model="gpt-4o", temperature=0)

response = model.invoke([
    {"role": "system", "content": "You are a helpful assistant. The output should be in Markdown"},
    {"role": "user", "content": "C'est quoi un Agent AI"}
])

response = model.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

print(display(Markdown(response.content)))

In [ ]:
from langchain_ollama import ChatOllama

llm_ollama = ChatOllama(model="llama3.2")

response_ollama = llm_ollama.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

print(display(Markdown(response_ollama.content)))

In [ ]:
load_dotenv(override=True)

from langchain_groq import ChatGroq

llm3 = ChatGroq(model="llama-3.3-70b-versatile")

resp3 = llm3.invoke([
    SystemMessage("You are a helpful assistant. The output should be in Markdown"),
    HumanMessage("C'est quoi un Agent AI")
])

print(display(Markdown(resp3.content)))

In [ ]:
system_message = """
Effectuez une analyse de sentiments basee sur les aspects des avis concernant
les ordinateurs portables presentes en entree.
Chaque avis peut comporter un ou plusieurs des aspects suivants : screen, keybloard et pad.
Pour chaque avis presente en entree :
- Identifiez la presence d'au moins un des trois aspects (screen, keybloard, pad).
- Attribuez une polarite de sentiment (positive, negative ou neutral) a chaque aspect.
  Organisez votre reponse dans un objet JSON avec les en-tetes suivants :
  - category:[liste des aspects]
  - polarity:[liste des polarites correspondantes pour chaque aspect]
Si l'un des aspects n'est pas present dans l'avis, tu supposes que la polarite est neutre.
"""

In [ ]:
zeroshot_prompt = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": """
        L'ecran est tres bon, mais je n'ai pas aime la souris. le clavier Ma fih Maytchaf
        """}
]

llm = ChatOpenAI(model="gpt-4o")
resp = llm.invoke(input=zeroshot_prompt)
print(resp.content)

In [ ]:
import json

resp_text_json = resp.content.replace("```json", "").replace("```", "")
print(resp_text_json)

result = json.loads(resp_text_json)
print(type(result))
print(result)
print(result['category'])
print(result['polarity'][0])

In [ ]:
from IPython.display import Image
import base64

llm4 = ChatOpenAI(model="gpt-4o")
llm_with_tools = llm4.bind_tools([
    {"type": "image_generation", "quality": "high"}
])

resp4 = llm_with_tools.invoke(input=[
    HumanMessage("Je veux une photo d'un chat qui code du java")
])

Image(base64.b64decode(resp4.content_blocks[0]['base64']))

In [ ]:
path = "rag.png"
Image(path)

In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

img = encode_image(path)

llm5 = ChatOpenAI(model="gpt-4o")
resp5 = llm5.invoke(input=[
    HumanMessage(content=[
        {"type": "text", "text": "Qu'est ce que tu vois dans cette image"},
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img}"}}
    ])
])

print(display(Markdown(resp5.content)))

In [ ]:
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import numpy as np
from datasets import load_dataset
import matplotlib.pyplot as plt

load_dotenv(override=True)

In [ ]:
corpus = load_dataset("imdb")
corpus

In [ ]:
train_df = corpus['train'].to_pandas()
train_df.info()

In [ ]:
train_df.label.value_counts()

In [ ]:
train_df['sentiment'] = np.where(train_df.label == 1, "positive", "negative")
train_df.sample(6)

In [ ]:
train_df.sentiment.value_counts()

In [ ]:
train_df.sentiment.hist()
plt.show()

In [ ]:
examples_df, gold_examples_df = train_test_split(
    train_df, test_size=0.2, random_state=42
)

(examples_df.shape, gold_examples_df.shape)

In [ ]:
columns = ['text', 'sentiment']
gold_examples = (
    gold_examples_df.loc[:, columns]
    .sample(20, random_state=42)
    .to_json(orient='records')
)

json.loads(gold_examples)[0]

In [ ]:
user_message_template = """```{movie_review}```"""

In [ ]:
zero_shot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.
"""

zero_shot_prompt = [{'role': 'system', 'content': zero_shot_system_message}]

In [ ]:
few_shot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.
"""

def create_examples(dataset, n=4):
    positive_reviews = (dataset.sentiment == 'positive')
    negative_reviews = (dataset.sentiment == 'negative')
    columns_to_select = ['text', 'sentiment']
    positive_examples = dataset.loc[positive_reviews, columns_to_select].sample(n)
    negative_examples = dataset.loc[negative_reviews, columns_to_select].sample(n)
    examples = pd.concat([positive_examples, negative_examples])
    randomized_examples = examples.sample(2*n, replace=False)
    return randomized_examples.to_json(orient='records')

examples = create_examples(examples_df, 2)
json.loads(examples)

In [ ]:
def create_prompt(system_message, examples, user_message_template):
    few_shot_prompt = [{'role': 'system', 'content': system_message}]
    for example in json.loads(examples):
        example_review   = example['text']
        example_sentiment = example['sentiment']
        few_shot_prompt.append({
            'role': 'user',
            'content': user_message_template.format(movie_review=example_review)
        })
        few_shot_prompt.append(
            {'role': 'assistant', 'content': f"{example_sentiment}"}
        )
    return few_shot_prompt

few_shot_prompt = create_prompt(few_shot_system_message, examples, user_message_template)
few_shot_prompt

In [ ]:
cot_system_message = """
Classify the sentiment of movie reviews presented in the input as 'positive' or 'negative'
Movie reviews will be delimited by triple backticks ``` in the input.
Answer only 'positive' or 'negative'
Do not explain your answer.

Instructions:
1. Carefully read the text of the review and think through the options for sentiment provided
2. Consider the overall sentiment of the review and estimate the probability of the review being positive

To reiterate, your answer should strictly only contain the label: positive or negative
"""

cot_few_shot_prompt = create_prompt(cot_system_message, examples, user_message_template)
cot_few_shot_prompt

In [ ]:
def evaluate_prompt(prompt, gold_examples, user_message_template, model):
    model_predictions, ground_truths = [], []
    for example in json.loads(gold_examples):
        gold_input = example['text']
        user_input = [{
            'role': 'user',
            'content': user_message_template.format(movie_review=gold_input)
        }]
        try:
            response = model.invoke(prompt + user_input)
            response_content = response.content if hasattr(response, 'content') else response

            if 'negative' in response_content.strip().lower():
                prediction = 'negative'
            elif 'positive' in response_content.strip().lower():
                prediction = 'positive'
            else:
                prediction = 'unknown'

            model_predictions.append(prediction)
            ground_truths.append(example['sentiment'])
            # print(prediction, example['sentiment'])
        except Exception as e:
            print(e)
            continue

    micro_f1 = f1_score(ground_truths, model_predictions, average="micro")
    return micro_f1

In [ ]:
gpt4o = ChatOpenAI(model="gpt-4o", temperature=0)

print("Zero-shot F1:", evaluate_prompt(zero_shot_prompt, gold_examples, user_message_template, gpt4o))

In [ ]:
print("Few-shot F1 :", evaluate_prompt(few_shot_prompt, gold_examples, user_message_template, gpt4o))

In [ ]:
print("CoT F1      :", evaluate_prompt(cot_few_shot_prompt, gold_examples, user_message_template, gpt4o))

In [ ]:
num_eval_runs = 10
few_shot_performance, cot_few_shot_performance = [], []

for _ in tqdm(range(num_eval_runs)):
    run_examples = create_examples(examples_df)
    run_few_shot = create_prompt(few_shot_system_message, run_examples, user_message_template)
    run_cot      = create_prompt(cot_system_message,      run_examples, user_message_template)

    few_shot_performance.append(
        evaluate_prompt(run_few_shot, gold_examples, user_message_template, gpt4o))
    cot_few_shot_performance.append(
        evaluate_prompt(run_cot, gold_examples, user_message_template, gpt4o))

In [ ]:
mean_fs  = np.array(few_shot_performance).mean()
std_fs   = np.array(few_shot_performance).std()
mean_cot = np.array(cot_few_shot_performance).mean()
std_cot  = np.array(cot_few_shot_performance).std()

print(f"Few-shot - f1_score Moyen = {mean_fs:.4f}, Standard Deviation = {std_fs:.4f}")
print(f"CoT      - f1_score Moyen = {mean_cot:.4f}, Standard Deviation = {std_cot:.4f}")

plt.figure(figsize=(9, 4))
plt.plot(few_shot_performance,     label="Few-shot", marker="o")
plt.plot(cot_few_shot_performance, label="CoT",      marker="s")
plt.xlabel("Run")
plt.ylabel("Micro-F1 Score")
plt.title("Performance Comparison: Few-shot vs CoT (10 runs)")
plt.legend()
plt.tight_layout()
plt.show()